In [1]:
"""
─────────
Factory functions that construct the four specialised agents:
  • TriageAgent      — classifies and routes requests
  • BillingAgent     — handles invoices, refunds, account queries
  • TechAgent        — handles outages, diagnostics, tickets
  • SummarizerAgent  — produces end-of-session summaries

Each agent shares:
  - Azure OpenAI backend (via OpenAIChatClient)
  - UserMemoryProvider + ConversationSummaryProvider
  - Full middleware stack (timing, security, rate-limit, tool-log, chat-audit)
"""


'\n─────────\nFactory functions that construct the four specialised agents:\n  • TriageAgent      — classifies and routes requests\n  • BillingAgent     — handles invoices, refunds, account queries\n  • TechAgent        — handles outages, diagnostics, tickets\n  • SummarizerAgent  — produces end-of-session summaries\n\nEach agent shares:\n  - Azure OpenAI backend (via OpenAIChatClient)\n  - UserMemoryProvider + ConversationSummaryProvider\n  - Full middleware stack (timing, security, rate-limit, tool-log, chat-audit)\n'

In [ ]:
# !pip install agent-framework


^C


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
fastmcp 3.0.2 requires rich>=13.9.4, but you have rich 13.7.1 which is incompatible.
openapi-core 0.19.5 requires werkzeug<3.1.2, but you have werkzeug 3.1.8 which is incompatible.
semantic-kernel 1.39.2 requires azure-ai-projects~=1.0.0b12, but you have azure-ai-projects 2.0.1 which is incompatible.
semantic-kernel 1.39.2 requires openai<2,>=1.98.0, but you have openai 2.30.0 which is incompatible.
semantic-kernel 1.39.2 requires pydantic!=2.10.0,!=2.10.1,!=2.10.2,!=2.10.3,<2.12,>=2.0, but you have pydantic 2.12.5 which is incompatible.
semantic-kernel 1.39.2 requires websockets<16,>=13, but you have websockets 16.0 which is incompatible.


  Using cached agent_framework_lab-1.0.0b251024-py3-none-any.whl.metadata (5.7 kB)
  Using cached uvicorn-0.41.0-py3-none-any.whl.metadata (6.7 kB)
  Using cached asyncio-4.0.0-py3-none-any.whl.metadata (994 bytes)
  Using cached azure_ai_inference-1.0.0b9-py3-none-any.whl.metadata (34 kB)
  Using cached foundry_local_sdk-0.5.1-py3-none-any.whl.metadata (6.1 kB)
Using cached uvicorn-0.41.0-py3-none-any.whl (68 kB)
   ---------------------------------------- 0.0/75.0 MB ? eta -:--:--
   ---------------------------------------- 0.0/75.0 MB ? eta -:--:--
   ---------------------------------------- 0.0/75.0 MB ? eta -:--:--
   ---------------------------------------- 0.3/75.0 MB ? eta -:--:--
   ---------------------------------------- 0.3/75.0 MB ? eta -:--:--
   ---------------------------------------- 0.3/75.0 MB ? eta -:--:--
   ---------------------------------------- 0.5/75.0 MB 446.5 kB/s eta 0:02:47
   ---------------------------------------- 0.5/75.0 MB 446.5 kB/s eta 0:02:47
   -

In [6]:
import os 
import re 
from typing import Any
import asyncio 
import time 
from collections import defaultdict
from collections.abc import Awaitable, Callable
from datetime import datetime, timezone 
from __future__ import annotations

from agent_framework import (AgentContext, AgentMiddleware,ChatContext, ChatMiddleware,FunctionMiddleware, FunctionInvocationContext)
from agent_framework import Agent 
from agent_framework.openai import OpenAIChatClient 
from agent_framework import AgentSession , ContextProvider, SessionContext 




# 1.  USER MEMORY PROVIDER
#     Remembers user name, account tier, and turn count.


In [3]:
class UserMemoryProvider(ContextProvider):
    """Short ter conversational memory that persists for the user identity 
    and account tier inside the AgentSession. State dict 
    before_run :  injects personalization instructions
    after_run : scans messages for "my name is X" and updates the session state with user name and account tier
    """
    SOURCE_ID = "user_memory"
    def __init__(self) -> None : 
        super().__init__(source_id=self.SOURCE_ID)
    async def before_run(self, *, agent : Any ,session: AgentSession , context: SessionContext, state: dict[str,Any]) -> None :
        turn = state.get("turn_count",0) + 1
        state["turn_count"] = turn
        lines : list[str] = [f"[Memory| Turn {turn}]"]
        name  = state.get("user_name")

        if name :
            lines.append(f"User name is {name}. Address them by name in responses.")
        else :
            lines.append("User name is unknown. Look for cues in the conversation to identify and remember the user name." )

        tier = state.get("user_tier")
        if tier == "premium" :
            lines.append("User is a premium customer. They are more likely to ask for refunds and expect faster resolution.")   

        elif tier == "standard" :
            lines.append("User is a standard customer. They may be more price sensitive and less likely to ask for refunds.")
        
        if state.get("open_ticket"):
            lines.append(f"Open ticket on file: {state['open_ticket']}. Reference it if relevant.")

        instruction = "\n".join(lines)
        context.add_system_instruction(instruction)
        print(f"Injected memory context: \n{instruction}\n")


    async def after_run(
        self, * , agent : Any, session : AgentSession, context : SessionContext,
        state : dict[str,Any]
    ) -> None:
        for msg in context.get_messages():
            text = (getattr(msg,"text",None) or "").lower()
            if not isinstance(text,str):
                continue
            tl = text.lower()

            # Extract name 
            match = re.search(r"my name is \s+([a-zA-Z]+)", tl)
            if match and not state.get("user_name"):
                name = match.group(1).capitalize()
                state["user_name"] = name 
                print(f"Extracted user name: {name} from message: {text}")

            # Extract tier from account lookup results in conversation
            if "premium" in tl :
                state.setdefault("user_tier","premium")
                print(f"Identified user as premium tier based on message: {text}")
            elif '"tier": "standard"' in tl or "standard tier" in tl:
                state.setdefault("user_tier","standard")
                print(f"Identified user as standard tier based on message: {text}")

            # remember if a ticket was created 
            ticket_match = re.search(r"(TKT-\d{4})", text)
            if ticket_match:
                ticket_id = ticket_match.group(1)
                state["open_ticket"] = ticket_id
                print(f"Identified open ticket {ticket_id} from message: {text}")
                

In [4]:
# Conversation Sunnary provider
class ConversationSummaryProvider(ContextProvider):
    """ 
    Keeps a rooling transcript of the last MAX_TURN turns and injects it as additional 
    context so every specialit agent knows what happend ealier in the conversation.
    """
    SOURCE_ID = "conv_summary"
    MAX_TURNS = 6 
    def __init__(self) -> None : 
        super().__init__(source_id=self.SOURCE_ID)
        self._transcript : list[str] = []
    
    def record(self,role : str, text : str) -> None :
        """ CAlled externally by the orchestrator after each turn"""
        entry = f"{role}: {text}"
        self._transcript.append(entry)
        # keep rolling windows 
        if len(self._transcript) > self.MAX_TURNS * 2 :
            self._transcript = self._transcript[-self.MAX_TURNS*2 :]

    async def before_run(
        self, *, agent : Any, session : AgentSession , context : SessionContext, state : dict[str,Any]
    ) -> None:
        if self._transcript:
            summary = "\n".join(self._transcript)
            instruction = f" Conversation summary  so far :\n {summary}\n End of summary. Use this context to inform your responses."
            context.extend_instructions(self.SOURCE_ID, instruction)
            print(f"Injected conversation summary context: \n{instruction}\n")
    async def after_run(
        self, *, agent: Any, session : AgentSession, context : SessionContext,
        state : dict[str,Any]
    )     -> None :
        # no need to scan for names or tiers here, just record the conversation for context 
        pass 
    
               

In [5]:
# Conversation Hook
# Application level lifecycle callbacks (not a framework type, but a pattern for cross cutting telemetry , audit logging, etc)

class ConversationHook : 
    """ Fires callbacks around each conversational turn 
    useful  for : telemetry, audit logging, etc
    """
    def __init__(self, name : str = "App") -> None :
        self.name = name 
        self._turn = 0
        self._log  : list[dict[str,Any]] = []

    # public hooks 
    def on_turn_start(self, user_input : str, agent_name : str) -> None :
        self._turn += 1
        entry = {
            "turn": self._turn,
            "agent": agent_name,
            "user_input": user_input,
            
        }
        self._log.append(entry)
        print(f"[{self.name} Hook] Starting turn {self._turn} with agent {agent_name} and user input: {user_input}")
        print(f"Current conversation log: {self._log}")

    def on_turn_end(self, response : str | None , elapsed_ms :float) -> None :
        preview : str = (response or "(no repsonse)").strip()[:100]
        print(f"[{self.name} Hook] Finished turn {self._turn} in {elapsed_ms:.2f} ms. Agent response preview: {preview}")
        if self._log : 
            self._log[-1]["response_preview"] = preview
            self._log[-1]["elapsed_ms"] = elapsed_ms
        print(f"Updated conversation log: {self._log}")
        print(f" Agent : {preview}")

    def on_error(self, error : Exception) -> None :
        print(f"[{self.name} Hook] Error occurred: {error}")
        if self._log :
            self._log[-1]["error"] = str(error)

    def on_security_block(self,reason  : str) -> None :
        print(f"[{self.name} Hook] Security block: {reason}")
        if self._log :
            self._log[-1]["security_block"] = reason
        print(f"\n [{self.name} Hook] Conversation log after security block: {self._log}\n security block reason: {reason}\n")

    def get_audit_log(self) -> list[dict[str,Any]] :
        return list(self._log)

    def print_audit_lof(self) -> None :
        print(f"\n[{self.name} Hook] Full conversation audit log:")
        for entry in self._log:
            print(entry)
        print(f"End of conversation audit log.\n")




"""
middleware.py
─────────────
All middleware implementations for the multi-agent system.

Microsoft Agent Framework supports three middleware types:
  1. AgentMiddleware  – wraps the entire agent.run() call
  2. FunctionMiddleware – wraps each individual tool call
  3. ChatMiddleware   – wraps the raw LLM inference call

Both function-based and class-based styles are demonstrated.
"""
